# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Standardize column headers
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [2]:
# Filter customers with total_claim_amount < 1000 and response == 'Yes'
filtered_df = df[(df['total_claim_amount'] < 1000) & (df['response'] == 'Yes')]
print(filtered_df.head())

    unnamed:_0 customer       state  customer_lifetime_value response  \
3            3  XL78013      Oregon             22332.439460      Yes   
8            8  FM55990  California              5989.773931      Yes   
15          15  CW49887  California              4626.801093      Yes   
19          19  NJ54277  California              3746.751625      Yes   
27          27  MQ68407      Oregon              4376.363592      Yes   

    coverage education effective_to_date employmentstatus gender  ...  \
3   Extended   College           1/11/11         Employed      M  ...   
8    Premium   College           1/19/11         Employed      M  ...   
15     Basic    Master           1/16/11         Employed      F  ...   
19  Extended   College           2/26/11         Employed      F  ...   
27   Premium  Bachelor           2/28/11         Employed      F  ...   

    number_of_open_complaints number_of_policies     policy_type  \
3                         0.0                  2  Corp

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [3]:
# Filter for respondents
yes_df = df[df['response'] == 'Yes']

# Aggregation table
analysis = yes_df.groupby(['policy_type', 'gender']).agg(
    avg_monthly_premium=('monthly_premium_auto', 'mean'),
    avg_clv=('customer_lifetime_value', 'mean'),
    avg_total_claim=('total_claim_amount', 'mean')
).round(2)

print(analysis)

                       avg_monthly_premium  avg_clv  avg_total_claim
policy_type    gender                                               
Corporate Auto F                     94.30  7712.63           433.74
               M                     92.19  7944.47           408.58
Personal Auto  F                     99.00  8339.79           452.97
               M                     91.09  7448.38           457.01
Special Auto   F                     92.31  7691.58           453.28
               M                     86.34  8247.09           429.53


3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [4]:
# Count customers per state
state_counts = df['state'].value_counts()

# Filter for states with more than 500 customers
top_states = state_counts[state_counts > 500]
print(top_states)

state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [5]:
# Aggregating min, max, and median for CLV
clv_stats = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(
    min_clv='min',
    max_clv='max',
    median_clv='median'
).round(2)

print(clv_stats)

                             min_clv   max_clv  median_clv
education            gender                               
Bachelor             F       1904.00  73225.96     5640.51
                     M       1898.01  67907.27     5548.03
College              F       1898.68  61850.19     5623.61
                     M       1918.12  61134.68     6005.85
Doctor               F       2395.57  44856.11     5332.46
                     M       2267.60  32677.34     5577.67
High School or Below F       2144.92  55277.45     6039.55
                     M       1940.98  83325.38     6286.73
Master               F       2417.78  51016.07     5729.86
                     M       2272.31  50568.26     5579.10


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [6]:
# Ensure effective_to_date is datetime and extract month
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])
df['month'] = df['effective_to_date'].dt.strftime('%B')

# Pivot table: states as rows, months as columns
policies_pivot = df.pivot_table(
    index='state',
    columns='month',
    values='customer',
    aggfunc='count'
)

print(policies_pivot)

C:\Users\lervi\AppData\Local\Temp\ipykernel_11176\2088714942.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])


month       February  January
state                        
Arizona          929     1008
California      1634     1918
Nevada           442      551
Oregon          1344     1565
Washington       425      463


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [7]:
# 1. Identify top 3 states by policy count
top_3_states = df['state'].value_counts().head(3).index

# 2. Filter DataFrame for top 3 states
df_top3 = df[df['state'].isin(top_3_states)]

# 3. Group by month and state to get count
policies_top3 = df_top3.groupby(['state', 'month'])['customer'].count().reset_index()
policies_top3.columns = ['state', 'month', 'policy_count']

print(policies_top3.sort_values(by=['state', 'policy_count'], ascending=[True, False]))

        state     month  policy_count
1     Arizona   January          1008
0     Arizona  February           929
3  California   January          1918
2  California  February          1634
5      Oregon   January          1565
4      Oregon  February          1344


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [8]:
# Calculate response rate (proportion of "Yes") per sales channel
channel_response = df.groupby('sales_channel')['response'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index(name='response_rate_%')

# Unpivot/Melt into long format as hinted
melted_response = pd.melt(
    channel_response,
    id_vars=['sales_channel'],
    value_vars=['response_rate_%'],
    var_name='metric',
    value_name='percentage'
)

print(melted_response.round(2))

  sales_channel           metric  percentage
0         Agent  response_rate_%       18.01
1        Branch  response_rate_%       10.79
2   Call Center  response_rate_%       10.32
3           Web  response_rate_%       10.89


External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here